In [144]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("../data/processed/cleaned_samsung_phones.csv")

df.head()

,model_name,series,launch_year,price_inr,ram_gb,storage_gb,processor_score,main_camera_mp,ultrawide_mp,telephoto_mp,front_camera_mp,ois,battery_mah,charging_w,refresh_rate_hz,screen_size_inch,weight_g,ai_features,waterproof,target_segment
0,Galaxy A06,A,2025,10999,4,64,3.8,50,2,0,8,No,5000,25,60,6.7,189,No,No,budget
1,Galaxy A15 5G,A,2024,17999,6,128,5.1,50,5,0,13,No,5000,25,90,6.5,200,No,No,budget
2,Galaxy A16 5G,A,2025,19999,8,128,5.7,50,8,0,13,No,5000,25,90,6.7,198,Basic AI,No,budget
3,Galaxy A25 5G,A,2024,24999,8,128,6.5,50,8,0,13,Yes,5000,25,120,6.5,197,Basic AI,Yes,budget
4,Galaxy A26 5G,A,2025,27999,8,256,6.9,64,8,0,13,Yes,5000,25,120,6.7,197,Basic AI,Yes,business


In [145]:
scaler = MinMaxScaler(feature_range=(0,10))

In [146]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0,10))

main_camera_norm = scaler.fit_transform(df[["main_camera_mp"]]).flatten()
ultrawide_norm = scaler.fit_transform(df[["ultrawide_mp"]]).flatten()
telephoto_norm = scaler.fit_transform(df[["telephoto_mp"]]).flatten()
front_camera_norm = scaler.fit_transform(df[["front_camera_mp"]]).flatten()
ai_norm = df["ai_features"].map({"No": 0, "Basic AI": 5, "Galaxy AI": 10}) / 10

camera_score_raw = (
    0.60 * main_camera_norm +
    0.15 * ultrawide_norm +
    0.10 * telephoto_norm +
    0.10 * front_camera_norm +
    0.05 * ai_norm
)

df["camera_score"] = scaler.fit_transform(camera_score_raw.to_frame())

In [147]:
processor_norm = scaler.fit_transform(df[["processor_score"]]).flatten()

ram_norm = scaler.fit_transform(df[["ram_gb"]]).flatten()
storage_norm = scaler.fit_transform(df[["storage_gb"]]).flatten()

df["performance_score"] = (
    0.6 * processor_norm +
    0.25 * ram_norm +
    0.15 * storage_norm
)

In [148]:
battery_norm = scaler.fit_transform(df[["battery_mah"]]).flatten()
charging_norm = scaler.fit_transform(df[["charging_w"]]).flatten()

df["battery_score"] = (
    0.7 * battery_norm +
    0.3 * charging_norm
)

In [149]:
refresh_norm = scaler.fit_transform(df[["refresh_rate_hz"]]).flatten()
screen_norm = scaler.fit_transform(df[["screen_size_inch"]]).flatten()

df["display_score"] = (
    0.6 * refresh_norm +
    0.4 * screen_norm
)

In [150]:
mapping = {
    "No":0,
    "Basic AI":5,
    "Galaxy AI":10
}

df["ai_score"] = df["ai_features"].map(mapping)

In [151]:
df["durability_score"] = df["waterproof"].map({
    "No":0,
    "Yes":10
})

In [152]:
capability_score = (
    0.25 * df["camera_score"] +
    0.25 * df["performance_score"] +
    0.20 * df["battery_score"] +
    0.15 * df["display_score"] +
    0.10 * df["durability_score"] +
    0.05 * df["ai_score"]
)

df["value_raw"] = capability_score / np.log1p(df["price_inr"])

df["value_score"] = df["value_raw"]

In [ ]:
value_scaler = MinMaxScaler(feature_range=(1,10))

df["value_score"] = value_scaler.fit_transform(
    df[["value_score"]]
)

df = df.drop(columns=["value_raw"], errors="ignore")

In [154]:
df[
[
    "model_name",
    "camera_score",
    "performance_score",
    "battery_score",
    "display_score",
    "durability_score",
    "ai_score",
    "value_score"
]
].head(30)

,model_name,camera_score,performance_score,battery_score,display_score,durability_score,ai_score,value_score
0,Galaxy A06,0.000000,0.000000,3.961538,1.428571,0,0,1.000000
1,Galaxy A15 5G,0.346558,1.889017,3.961538,3.857143,0,0,2.244093
2,Galaxy A16 5G,0.482792,2.886329,3.961538,4.428571,0,5,3.090714
3,Galaxy A25 5G,0.482792,3.660522,3.961538,6.857143,10,5,5.166896
4,Galaxy A26 5G,1.125239,4.476190,3.961538,7.428571,10,5,5.720133
5,Galaxy A35 5G,1.677820,5.056836,3.961538,7.142857,10,10,6.282205
6,Galaxy A36 5G,1.746654,5.443932,5.961538,7.428571,10,10,6.969571
7,Galaxy A55 5G,1.792543,6.857911,5.961538,7.142857,10,10,7.267972
8,Galaxy M15 5G,0.346558,2.179339,6.653846,3.857143,0,0,3.162182
9,Galaxy M35 5G,0.482792,5.153610,6.653846,7.142857,0,5,5.061673


In [155]:
df.to_csv(
    "../data/processed/engineered_dataset.csv",
    index=False
)

print("Feature engineering completed.")

Feature engineering completed.


In [156]:
df["main_camera_mp"].value_counts()

main_camera_mp
50     22
64      3
200     3
108     2
Name: count, dtype: int64